<a href="https://colab.research.google.com/github/raz0208/Retrieval-Augmented-Generation-RAG/blob/main/Retrieval_Augmented_Generation_(RAG)_from_Scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Retrieval-Augmented Generation(RAG) From Scratch
#### A traditional and basic RAG implementation

In this notebook, we will build a Retrieval-Augmented Generation(RAG) pipeline from scratch without using any popular libraries such as Langchain or Llamaindex.

RAG is a technique that retrieves related documents to the user's question, combines them with LLM-base prompt, and sends them to LLMs like GPT to produce more factually accurate generation.

## Lets Split RAG Pipeline into 5 parts:

1. Data loading
2. Chunking and Embedding
3. Vector Store
4. Retrieval & Prompt Preparation
5. Answer Generation

## Step 1: Install Dependences and Import Required Libraries

In [31]:
!pip install transformers scikit-learn docx2txt datasets nltk lancedb openai -q

## Set OPENAI API KEY as env variable

In [32]:
# Setp OpenAI API Key
import os

# Copy you API key here
os.environ["OPENAI_API_KEY"] = "sk-xxx"

In [33]:
# Load and read data pdf/txt

# !wget link
!wget https://raw.githubusercontent.com/lancedb/vectordb-recipes/main/tutorials/RAG-from-Scratch/lease.txt

# Open and read the file - change the name of file related to the download link
with open("lease.txt", "r") as file:
    text_data = file.read()

--2026-04-29 16:23:24--  https://raw.githubusercontent.com/lancedb/vectordb-recipes/main/tutorials/RAG-from-Scratch/lease.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5248 (5.1K) [text/plain]
Saving to: ‘lease.txt.2’

lease.txt.2         100%[===================>]   5.12K  --.-KB/s    in 0s      

2026-04-29 16:23:24 (77.4 MB/s) - ‘lease.txt.2’ saved [5248/5248]



In [34]:
# text_data
print(text_data[:500])

EX-10 2 elmonteleaseforfiling.htm MATERIAL CONTRACT
COMMERCIAL LEASE AGREEMENT



THIS LEASE AGREEMENT is made and entered into on December 1, 2013, by and between Temple CB, LLC, whose address is 4350 Temple City Boulevard, El Monte, California 91731 (hereinafter referred to as "Landlord"), and Okra Energy, Inc., whose address is 4350 Temple City Boulevard, El Monte, California 91731 (hereinafter referred to as "Tenant").



ARTICLE I - GRANT OF LEASE



Landlord, in consideration of the rents 


## Data Chunking and Embedding

In [35]:
import nltk
import re
from nltk.tokenize import sent_tokenize

nltk.download("punkt")

# Recursive Function To Split Texts and Creat Chunks
def recursive_text_splitter(text, max_chunk_length=1000, overlap=100):
    """
    Helper function for chunking text recursively
    """
    # Initialize result
    result = []

    current_chunk_count = 0
    separator = ["\n", " "]
    _splits = re.split(f"({separator})", text)
    splits = [_splits[i] + _splits[i + 1] for i in range(1, len(_splits), 2)]

    for i in range(len(splits)):
        if current_chunk_count != 0:
            chunk = "".join(
                splits[
                    current_chunk_count
                    - overlap : current_chunk_count
                    + max_chunk_length
                ]
            )
        else:
            chunk = "".join(splits[0:max_chunk_length])

        if len(chunk) > 0:
            result.append("".join(chunk))
        current_chunk_count += max_chunk_length

    return result

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [36]:
# Split the text using the recursive character text splitter
chunks = recursive_text_splitter(text_data, max_chunk_length=100, overlap=10)
print("Number of Chunks: ", len(chunks))
print("Number of Token Per Each Chunk:", len(chunks[0].split()))

Number of Chunks:  11
Number of Token Per Each Chunk: 79


## Embedding By Using a Pre-Trained LLM Model

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

# Choose a pre-trained model (e.g., BERT, RoBERTa, etc.)
# Load the tokenizer and model
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# Define Embedder Function
def embedder(chunk):
    """
    Helper function to embed chunk of text
    """

    # Tokenize the input text
    tokens = tokenizer(chunk, return_tensors="pt", padding=True, truncation=True)

    # Get the model's output (including embeddings)
    with torch.no_grad():
      model_output = model(**tokens)

    # Extract the embeddings
    embeddings = model_output.last_hidden_state[:, 0, :]
    embed = embeddings[0].numpy()

    return embed

In [38]:
# Embed all the chunks of text
embeds = []
for chunk in chunks:
    embed = embedder(chunks)
    embeds.append(embed)

## Store All Embeddings In A Vector Database

In [39]:
# Insert text chunks with their embeddings

import lancedb

# Function to prepare data to insert in LanceDB
def prepare_data(chunks, embeddings):
    """
    Helper function to prepare data to insert in LanceDB
    """
    data = []
    for chunk, embed in zip(chunks, embeddings):
        temp = {}
        temp["text"] = chunk
        temp["vector"] = embed
        data.append(temp)
    return data

# Functio to insert data in LanceDB
def lanceDBConnection(chunks, embeddings):
    """
    LanceDB insertion
    """
    db = lancedb.connect("/tmp/lancedb")
    data = prepare_data(chunks, embeddings)
    table = db.create_table(
        "scratch",
        data=data,
        mode="overwrite",
    )
    return table


table = lanceDBConnection(chunks, embeds)

## Retriever & Prompt preparation

In [40]:
# Retriever
k = 5
question = "What is issue date of lease?"

# Embed Question
query_embedding = embedder(question)

# Semantic Search
result = table.search(query_embedding).limit(5).to_list()

In [41]:
context = [r["text"] for r in result]
context

[' 2 elmonteleaseforfiling.htm MATERIAL CONTRACT\nCOMMERCIAL LEASE AGREEMENT\n\n\n\nTHIS LEASE AGREEMENT is made and entered into on December 1, 2013, by and between Temple CB, LLC, whose address is 4350 Temple City Boulevard, El Monte, California 91731 (hereinafter referred to as "Landlord"), and Okra Energy, Inc., whose address is 4350 Temple City Boulevard, El Monte, California 91731 (hereinafter referred to as "Tenant").\n\n\n\nARTICLE I - GRANT OF LEASE\n\n\n\nLandlord, in consideration of the rents to be paid and the covenants',
 ' consideration of the rents to be paid and the covenants and agreements to be performed and observed by the Tenant, does hereby lease to the Tenant and the Tenant does hereby lease and take from the Landlord the property described in Exhibit "A" attached hereto and by reference made a part hereof (the "Leased Premises"), together with, as part of the parcel, all improvements located thereon.\n\n\n\nARTICLE II - LEASE TERM\n\n\n\nSection l.  Term of Leas

In [42]:
# Context Prompt

base_prompt = """You are an AI assistant. Your task is to understand the user question, and provide an answer using the provided contexts. Every answer you generate should have citations in this pattern  "Answer [position].", for example: "Earth is round [1][2].," if it's relevant.

Your answers are correct, high-quality, and written by an domain expert. If the provided context does not contain the answer, simply state, "The provided context does not have the answer."

User question: {}

Contexts:
{}
"""

## Answer Generation

In [ ]:
from openai import OpenAI
client = OpenAI()

prompt = base_prompt.format(question, context)

response = client.chat.completions.create(
    model="gpt-4o-mini",   # or "gpt-4o"
    temperature=0,
    messages=[
        {"role": "system", "content": "You answer questions using given context."},
        {"role": "user", "content": prompt},
    ],
)

print(response.choices[0].message.content)